# Demo – Mouse Dynamics Authentication | Pouzivatel 80

- nacitanie natrenovaneho XGBoost modelu pre pouzivatela 80 (median EER) 
- spustenie predikcie na testovacich datach
- SHAP analyza

## 1. Import kniznic a funkcii

In [ ]:
import os
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_curve, auc,
    accuracy_score, precision_score, recall_score,
)
from IPython.display import display

In [ ]:
USER_ID    = 80
SAMPLE_DIR = os.path.dirname(os.path.abspath("demo_user80.ipynb"))
MODEL_PATH = os.path.join(SAMPLE_DIR, "user_80.json")
TEST_CSV   = os.path.join(SAMPLE_DIR, "test_sample80.csv")

os.chdir(SAMPLE_DIR)

In [ ]:
_xai_path = os.path.abspath(os.path.join(SAMPLE_DIR, "..", "src", "classifier", "binary_xgb-xai.py"))
_spec = importlib.util.spec_from_file_location("binary_xgb_xai", _xai_path)
_xai  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_xai)

plot_shap_summary            = _xai.plot_shap_summary
plot_shap_local_explanations = _xai.plot_shap_local_explanations
print("Funkcie nacitane")

## 2. Nacitanie modelu a dat

In [ ]:
model = XGBClassifier()
model.load_model(MODEL_PATH)

feature_names = model.get_booster().feature_names

In [ ]:
df_raw = pd.read_csv(TEST_CSV, header=None)
df_raw.columns = ["_idx", "user_id"] + list(feature_names) + ["csv_file"]
df_raw = df_raw.drop(columns=["_idx"])

df_raw.head(3)

## 3. Predikcia a vyhodnotenie

In [ ]:
df_raw["label"] = (df_raw["user_id"] == USER_ID).astype(int)

X_test = df_raw[list(feature_names)].copy()
y_test = df_raw["label"].copy()
n_pos  = int(y_test.sum())

y_proba = model.predict_proba(X_test)[:, 1]

score_summary = pd.DataFrame({
    "score": y_proba.round(4),
    "label": y_test.values,
})
display(score_summary)
print(f"Priemerna uspesnost: {y_proba.mean():.4f}")

## 4. SHAP analyza

### 4a. Priprava TreeExplainera

In [ ]:
if hasattr(model, "base_score") and isinstance(model.base_score, str):
    model.base_score = 0.5

booster     = model.get_booster()
explainer   = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_test)

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = float(expected_value[-1])

print(f"SHAP hodnoty vypocitane  |  tvar: {shap_values.shape}")
print(f"Expected value (base):  {expected_value:.4f}")

### 4b. Force Plot

Staticke matplotlib renderovanie (viditelne aj na GitHube).

In [ ]:
pos_idx = np.where(y_test.values == 1)[0]

if len(pos_idx) > 0:
    i = pos_idx[0]
    sv_top_mask = np.argsort(np.abs(shap_values[i]))[-15:]
    
    shap.force_plot(
        expected_value,
        shap_values[i][sv_top_mask],
        X_test.iloc[i].iloc[sv_top_mask],
        feature_names=[feature_names[j] for j in sv_top_mask],
        matplotlib=True,
        show=True,
        figsize=(18, 3),
        text_rotation=15,
    )
else:
    print("Ziadne pozitivne vzorky.")

In [ ]:
# Force plot pre dalsiu vzorku
if len(pos_idx) > 1:
    i = pos_idx[len(pos_idx) // 2]
    sv_top_mask = np.argsort(np.abs(shap_values[i]))[-15:]
    
    shap.force_plot(
        expected_value,
        shap_values[i][sv_top_mask],
        X_test.iloc[i].iloc[sv_top_mask],
        feature_names=[feature_names[j] for j in sv_top_mask],
        matplotlib=True,
        show=True,
        figsize=(18, 3),
        text_rotation=15,
    )

### 4c. Decision Plot

In [ ]:
plt.figure(figsize=(12, 8))
shap.decision_plot(
    expected_value,
    shap_values,
    X_test,
    highlight=list(range(len(pos_idx))),
    show=False,
)
plt.title(
    f"Decision Plot - Pouzivatel {USER_ID}  |  {n_pos} autentifikovanych vzoriek",
    fontsize=12,
)
plt.tight_layout()
plt.show()

### 4d. Summary Plot

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.title(f"SHAP Summary - Pouzivatel {USER_ID}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()